In [1]:
import sys
import os

# For notebooks, use getcwd(); for scripts, use __file__
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Running in a notebook
    notebook_dir = os.getcwd()

sys.path.insert(0, notebook_dir)

from agents.orchestrator import analyze_user_input, update_state_from_analysis
from agents.specialist import run_activities_specialist, run_logistics_specialist
from agents.verifier import verify_and_format_itinerary, format_complete_itinerary
from state import TravelState


In [2]:
state = TravelState()
task_json = {
  "task_id": "task_123",
  "intent": "update_constraints",
  "updated_constraints": {
    "origin": "BOS",
    "destination": "SEA",
    "start_date": "2026-05-05",
    "end_date": "2026-05-12"
  },
  "delegation": "logistics",
  "response_to_user": "I will search for flights...",
}
user_input = None #"want to change my trip to Seattle from May 6th to May 13th"

analysis = analyze_user_input(user_input, state, task_json=task_json)
state = update_state_from_analysis(state, analysis)
delegation = analysis.get("delegation", "none")

In [3]:
 # 3. Delegation
if delegation == "none":
    response = analysis.get(
        "response_to_user",
        "I'm still missing some info to start planning.",
    )
    print(f"\nNomad: {response}")
    state.messages.append({"role": "assistant", "content": response})

In [4]:

# If we reach here, we need to run specialists
draft_components = []
all_search_results = {
    "flights": [],
    "hotels": [],
    "activities": [],
}
# Accumulate specialist contexts for verifier
specialist_contexts = {
    "logistics": None,
    "activities": None,
}
constraints_str = state.constraints.model_dump_json(indent=2)

if delegation in ["logistics", "both"]:
    print("\n[Specialist - Logistics] Searching for flights & hotels...")
    try:
        # NEW: Unpack all 3 return values
        logistics_draft, logistics_searches, logistics_context = run_logistics_specialist(
            constraints_str, 
            task_id=state.task_id
        )
        draft_components.append("--- LOGISTICS ---\n" + logistics_draft)
        
        # Accumulate search results
        all_search_results["flights"].extend(logistics_searches.get("flights", []))
        all_search_results["hotels"].extend(logistics_searches.get("hotels", []))
        
        # Save specialist context for verifier
        specialist_contexts["logistics"] = logistics_context
        
        print(f"\n[Logistics Specialist Summary]")
        print(f"  Search Coverage: {logistics_context['search_coverage']}")
        print(f"  Total Searches: {logistics_context['total_searches']}")
        print(f"  Tool Calls: {len(logistics_context['tool_calls_summary'])}")
        
    except Exception as e:
        print(f"[Specialist Error] {e}")
        import traceback
        traceback.print_exc()



[Specialist - Logistics] Searching for flights & hotels...
  [Specialist Turn 1] Thinking...
  🔧 Executing tool search_flights with arguments {'origin': 'BOS', 'destination': 'SEA', 'departure_date': '2026-05-05', 'return_date': '2026-05-12', 'task_id': 'task_123', 'turn': 0}
[Cache Hit] google_flights
🍽️  Flights results: 3 options
[Saved] Candidate -> c:\GIT\aiagent\nomad\src\agents\..\search_candidates\task_123\flights_candidates.json (8 total)
  🔧 Executing tool search_hotels with arguments {'location': 'Seattle, WA', 'check_in': '2026-05-05', 'check_out': '2026-05-12', 'adults': 1, 'task_id': 'task_123', 'turn': 0}
[Cache Hit] google_hotels
🍽️  Hotels results: 5 options
[Saved] Candidate -> c:\GIT\aiagent\nomad\src\agents\..\search_candidates\task_123\hotels_candidates.json (8 total)
  [Specialist Turn 2] Thinking...

[Logistics Specialist Summary]
  Search Coverage: {'flights': 6, 'hotels': 5, 'activities': 0}
  Total Searches: 11
  Tool Calls: 2


In [5]:
 # 4. Verifier: check draft against constraints
print("\n[Verifier] Checking itinerary against constraints...")
full_draft = "\n\n".join(draft_components)

print(f"\n[Debug] Search Results Found:")
print(f"  - Flights: {len(all_search_results['flights'])} items")
print(f"  - Hotels: {len(all_search_results['hotels'])} items")
print(f"  - Activities: {len(all_search_results['activities'])} items")

try:
    verification = verify_and_format_itinerary(
        full_draft, 
        constraints_str,
        task_id=state.task_id,
        search_results=all_search_results  # Pass search results to verifier
    )

    if verification.get("is_valid"):
        final_response = verification.get(
            "final_message_to_user", "Here is your plan!"
        )
        print("\nNomad:\n")
        print(final_response)
        state.messages.append({"role": "assistant", "content": final_response})
    else:
        issues = verification.get("issues", [])
        print(
            f"\nNomad: I built a plan, but it violates some constraints:\n{issues}"
        )
        print("I will need to revise. Let me know how you'd like to adjust.")

except Exception as e:
    print(f"Error during verification: {e}")


[Verifier] Checking itinerary against constraints...

[Debug] Search Results Found:
  - Flights: 6 items
  - Hotels: 5 items
  - Activities: 0 items
[Saved] Verification result -> c:\GIT\aiagent\nomad\src\agents\..\verification_results\task_123_verification.json

Nomad:

🎉 Your Seattle adventure is confirmed!

**✈️ FLIGHTS - $297**
Outbound: Delta DL 332 on May 5th
• Departs Boston (BOS) at 11:10 AM
• Arrives Seattle (SEA) at 2:23 PM local time
• Nonstop flight (6h 13m) on Airbus A321neo
• Includes: Free Wi-Fi, power outlets, Live TV
• 27% lower carbon emissions than typical
• Note: This flight is sometimes delayed 30+ min

Return: May 12th from Seattle to Boston (included in price)

**🏨 ACCOMMODATION - $634 (7 nights)**
BDH1 - Napoleons' Sweet Suite
• Ballard neighborhood micro studio
• Rating: 4.4/5 from 344 reviews
• Check-in: May 5 at 3:00 PM | Check-out: May 12 at 10:00 AM
• Full kitchen, washer, free Wi-Fi, cable TV
• Free cancellation until April 5, 2026
• 35 min taxi to airpor

In [ ]:
# 5. Evaluator: Assess the verified plan against task requirements
# Import evaluator
from tools.evaluator import NomadEvaluator
from tools.plan_schema import validate_plan

print("\n" + "="*70)
print("[EVALUATOR] Assessing plan quality and constraint satisfaction...")
print("="*70)

# Collect tool logs from specialist contexts
tool_logs = []
if specialist_contexts.get("logistics"):
    tool_logs.extend(specialist_contexts["logistics"].get("tool_calls_summary", []))
if specialist_contexts.get("activities"):
    tool_logs.extend(specialist_contexts["activities"].get("tool_calls_summary", []))

print(f"\n[Evaluator] Tool calls collected: {len(tool_logs)}")

# Validate schema compliance first
if verification.get("is_valid"):
    plan_to_evaluate = verification.get("itinerary", verification)
    
    schema_validation = validate_plan(plan_to_evaluate)
    print(f"\n[Schema Validation]")
    print(f"  Valid: {schema_validation['is_valid']}")
    print(f"  Score: {schema_validation['score']:.2f}/1.0")
    if schema_validation['errors']:
        print(f"  Errors: {schema_validation['errors']}")
    if schema_validation['missing_fields']:
        print(f"  Missing Fields: {schema_validation['missing_fields']}")
    
    # Initialize evaluator - NO task_file required!
    evaluator = NomadEvaluator()
    
    # Prepare hard constraints from state
    hard_constraints = [
        f"Origin: {state.constraints.origin}",
        f"Destination: {state.constraints.destination}",
        f"Duration: {(state.constraints.end_date - state.constraints.start_date).days} nights"
    ]
    
    # Run evaluation - DIRECT, no task lookup
    try:
        eval_result = evaluator.evaluate(
            agent_output=plan_to_evaluate,
            task_id=state.task_id,
            hard_constraints=hard_constraints,
            expected_tools=["flight_search", "hotel_search", "activities_search"],
            tool_logs=tool_logs
        )
        
        print(f"\n[EVALUATION RESULTS]")
        print(f"  Overall Score: {eval_result['overall_score']:.3f}/1.0")
        print(f"  ├─ Schema Compliance: {eval_result['schema_compliance']:.3f} (25% weight)")
        print(f"  ├─ CSR (Constraints): {eval_result['csr_score']:.3f} (35% weight)")
        print(f"  ├─ Tool Accuracy: {eval_result['tool_accuracy']:.3f} (20% weight)")
        print(f"  └─ Itinerary Validity: {'✓' if not eval_result['conflict_report']['has_conflicts'] else '✗'} (20% weight)")
        
        print(f"\n[Constraint Breakdown]")
        for constraint, met in eval_result['constraint_breakdown'].items():
            status = "✓" if met else "✗"
            print(f"  {status} {constraint}")
        
        if eval_result['conflict_report']['errors']:
            print(f"\n[Conflicts Detected]")
            for error in eval_result['conflict_report']['errors']:
                print(f"  ⚠ {error}")
        
        if eval_result['conflict_report']['warnings']:
            print(f"\n[Warnings]")
            for warning in eval_result['conflict_report']['warnings']:
                print(f"  ℹ {warning}")
        
        print(f"\n[FINAL VERDICT]")
        if eval_result['overall_score'] >= 0.8:
            print(f"  ✓ EXCELLENT PLAN (Score: {eval_result['overall_score']:.1%})")
        elif eval_result['overall_score'] >= 0.6:
            print(f"  ~ ACCEPTABLE PLAN (Score: {eval_result['overall_score']:.1%})")
        else:
            print(f"  ✗ NEEDS REVISION (Score: {eval_result['overall_score']:.1%})")
        
        # Store evaluation result in state for later use
        state.evaluation_result = eval_result
        
    except Exception as e:
        print(f"[Evaluation Error] {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"\n[Skipped] Plan is invalid, skipping evaluation")
    print(f"  Issues: {verification.get('issues', [])}")

ModuleNotFoundError: No module named 'plan_schema'

In [ ]:
# Quick Example: Use Evaluator directly without task_file
# ========================================================
# 以下演示新的 evaluator API - 直接调用，无需 task_file

print("\n" + "="*70)
print("[DEMO] Evaluator Usage - Direct API (No task_file needed)")
print("="*70)

# Create evaluator WITHOUT task_file
evaluator_demo = NomadEvaluator()

# Evaluate any plan directly
if hasattr(state, 'evaluation_result'):
    result = state.evaluation_result
    print(f"\n✓ Already evaluated above. Final Score: {result['overall_score']:.1%}")
    print(f"\nUsage example for standalone evaluation:")
    print("""
from tools.evaluator import NomadEvaluator

# No task_file needed!
evaluator = NomadEvaluator()

# Evaluate directly
result = evaluator.evaluate(
    agent_output=my_plan_dict,
    task_id="my_task_001",  # optional
    hard_constraints=["Budget: $3000", "Destination: Paris"],  # optional
    expected_tools=["flight_search", "hotel_search"],  # optional
    tool_logs=[...]  # optional
)

print(f"Score: {result['overall_score']:.1%}")
    """)
else:
    print("Run Cell 5 first to generate evaluation_result")